# Knowledge-editing AMD-Llama-135M with MARV (Colab T4 or CPU)

`amd/AMD-Llama-135m` is a Llama-2-architecture model (12 layers, hidden 768,
intermediate 2048, Llama-2 tokenizer) trained on ~670B tokens of SlimPajama +
Project Gutenberg. It loads as `LlamaForCausalLM`, so MARV's `LlamaStyleFFN`
adapter handles it with no special-casing.

**Why this model for an editing demo:** it is tiny. Extraction is instant,
`build_down_meta` over its 32k vocab is instant on CPU, and a **300-probe
battery runs in seconds** -- which is the whole point of the exercise below:
you cannot estimate collateral damage from 8 control probes, and here you can
afford 120+.

**Tokenizer gotcha (important).** AMD-Llama-135M was pretrained with **no BOS
(`<s>`) token** -- its model card uses `add_special_tokens=False` everywhere.
MARV tokenizes with the transformers default (BOS on), which sends this
particular model out of distribution and makes *every* prompt predict junk
(`'\n'`, `'ł'`, ...). The load cell below sets `tok.add_bos_token = False` to
fix it globally. Without that line the "pick a fact" cell finds nothing and
asserts.

**Honest caveat.** It is a *base* model (no chat template -- every prompt here
is plain completion) with weak factual recall. Even with the BOS fix it does
*not* reliably know every capital, so the notebook **probes a list of candidate
facts first and picks one the model actually gets right at rank 1**. If none
pass, that's a signal to use `Qwen/Qwen2.5-0.5B` instead (same notebook, just
change the model name) -- the method transfers, the 135M numbers are just
noisier.

Also runs the base-vs-code weight diff: `amd/AMD-Llama-135m` vs
`amd/AMD-Llama-135m-code` (StarCoder-Python finetune, same 12 layers).

Runtime: T4 GPU is fine; CPU works too.

In [ ]:
!pip install -q 'transformers>=4.40' accelerate safetensors matplotlib
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

## Load + extract

In [ ]:
import math, torch, numpy as np, gc, marv
from transformers import LlamaForCausalLM, AutoTokenizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE = 'amd/AMD-Llama-135m'
CODE = 'amd/AMD-Llama-135m-code'
DTYPE = torch.float16 if device == 'cuda' else torch.float32

tok = AutoTokenizer.from_pretrained(BASE)
# AMD-Llama-135M was pretrained on raw concatenated text with NO <s> (BOS) token
# -- its model card uses add_special_tokens=False everywhere. MARV's run_battery /
# context helpers tokenize with the transformers default (add BOS), which pushes
# this model out of distribution: every prompt then predicts junk ('\n', 'ł', ...).
# Disabling BOS on the tokenizer fixes it globally (tok is reused below).
tok.add_bos_token = False
model = LlamaForCausalLM.from_pretrained(BASE, torch_dtype=DTYPE).to(device).eval()
print(model.config.num_hidden_layers, 'layers  hidden', model.config.hidden_size,
      ' intermediate', model.config.intermediate_size, ' vocab', model.config.vocab_size)

# sanity check the fix: this base model does know "The capital of France is Paris"
_c = marv.run_battery(model, tok, [marv.Probe('The capital of France is', 'Paris', ('t',))], device=device)
print(f"sanity: 'The capital of France is' -> {_c.rows[0].top1!r} (target rank {_c.rows[0].target_rank})")

vindex = marv.extract(model, model_name=BASE)
marv.build_down_meta(vindex, device=device)      # 32k vocab -> instant
vindex.save('/content/amd-135m-base.vindex.npz')
print('bands:', vindex.layer_bands)

## Pick a fact this model actually knows

Editing a fact the model gets wrong unedited is meaningless. Probe a handful of
`country -> capital` facts and keep the first one the base model predicts at
**rank 1**.

In [ ]:
CANDIDATES = [
    ('France',  'Paris',  ['Italy', 'Spain', 'Germany', 'Portugal', 'Belgium']),
    ('Germany', 'Berlin', ['France', 'Italy', 'Poland', 'Austria', 'Belgium']),
    ('Italy',   'Rome',   ['France', 'Spain', 'Greece', 'Austria', 'Portugal']),
    ('Japan',   'Tokyo',  ['China', 'Thailand', 'Vietnam', 'Indonesia']),
    ('Russia',  'Moscow', ['Poland', 'Ukraine', 'Finland', 'Sweden']),
    ('China',   'Beijing',['Japan', 'Vietnam', 'Mongolia', 'Nepal']),
    ('Spain',   'Madrid', ['France', 'Italy', 'Portugal', 'Belgium']),
]

check = marv.run_battery(
    model, tok,
    [marv.Probe(f'The capital of {c} is', cap, ('t',)) for c, cap, _ in CANDIDATES],
    device=device,
)
for row in check.rows:
    flag = 'OK' if row.target_rank == 1 else f'r{row.target_rank}'
    print(f'  [{flag:>4}]  {row.prompt:<30} -> {row.top1!r:>12}   p={row.target_prob:.3f}')

known = [(c, cap, nb) for (c, cap, nb), row in zip(CANDIDATES, check.rows) if row.target_rank == 1]
assert known, ("AMD-135M got none of these capitals at rank 1. Add more candidates, "
               "or rerun this notebook with BASE = 'Qwen/Qwen2.5-0.5B'.")
COUNTRY, CAPITAL, NEIGHBOURS = known[0]
print(f'\n>>> editing  {COUNTRY} -> {CAPITAL}    neighbours: {NEIGHBOURS}')

## Browse + locate the constellation

`describe_entity` is the bare-embedding view; `constellation(..., model=)` queries
the real hidden state (differenced against a baseline), which is sharper.

In [ ]:
for entity in [COUNTRY, CAPITAL]:
    print(f'=== {entity} ===')
    for r in marv.describe_entity(vindex, tok, entity, k_features=3):
        print('  ', r)

pool = marv.constellation(vindex, tok, COUNTRY, model=model,
                          prompt=f'The capital of {COUNTRY} is',
                          baseline_prompt='The capital of',
                          per_layer=8, device=device)
print('\ncontextual constellation (top 12):')
for r in pool[:12]:
    print(f'  L{r.layer:>2} f{r.feature:<5} sim={r.sim:.2f}  -> {r.tokens[:3]}')

## Why 8 controls is not enough

Collateral rate is a **proportion**; its standard error is `sqrt(p(1-p)/n)`.
At n=8 and a true 10% rate the error bar (+/- 0.11) is bigger than the number
you're trying to measure. Below: the *same edit*, scored against ~8 controls and
against `marv.capital_edit_battery()` -- 4 target rephrasings, the neighbour
capitals, and `broad_controls()` (~110 probes across 6 sub-domains) with the
target + neighbours removed.

In [ ]:
P = marv.Probe

narrow = [
    P(f'The capital of {COUNTRY} is', CAPITAL, ('target',)),
    P(f'The capital city of {COUNTRY} is', CAPITAL, ('target',)),
    *marv.capital_probes(NEIGHBOURS[:2], tags=('neighbour', 'capital')),
    P('Water is made of hydrogen and', 'oxygen', ('control',)),
    P('The opposite of hot is', 'cold', ('control',)),
    P('Two plus two equals', 'four', ('control',)),
    P('The cat sat on the', 'mat', ('control',)),
    P('The sky is', 'blue', ('control',)),
    P('The past tense of go is', 'went', ('control',)),
    P('Bees make', 'honey', ('control',)),
]

wide = marv.capital_edit_battery(COUNTRY, CAPITAL, neighbours=NEIGHBOURS)
print('narrow:', len(narrow), ' wide:', len(wide))

## Baseline filter

Keep only probes the model already gets right (rank <= 3). On a 135M base model
a lot of the harder controls won't survive -- what's left is still far more
than 8, and the target is guaranteed in (we picked it for rank 1 above).

In [ ]:
def keep_known(battery, rank_max=3):
    r = marv.run_battery(model, tok, battery, device=device)
    ok = {row.prompt for row in r.rows if row.target_rank <= rank_max}
    return [p for p in battery if p.prompt in ok]

narrow_k = keep_known(narrow)
wide_k = keep_known(wide)

n_target = sum('target' in p.tags for p in wide_k)
n_ctrl = sum('control' in p.tags for p in wide_k)
assert n_target > 0, "no target probe survived the filter -- lower rank_max or repick the fact"

from collections import Counter
by_dom = Counter(t for p in wide_k for t in p.tags
                 if t not in ('control', 'target', 'neighbour', 'capital'))
print(f'narrow kept {len(narrow_k)}/{len(narrow)}')
print(f'wide   kept {len(wide_k)}/{len(wide)}   ({n_target} target, {n_ctrl} control)')
print('controls by sub-domain:', dict(by_dom))

## The causal constellation, then one edit

In [ ]:
target_probes = [p for p in wide_k if 'target' in p.tags]
ranked = marv.rank_by_ablation_effect(
    model, tok, [(r.layer, r.feature) for r in pool[:30]], target_probes, device=device)
for (L, f), drop in ranked[:12]:
    tks, _ = marv.describe_feature(vindex, L, f, k=3)
    print(f'  L{L:>2} f{f:<5} drop={drop:+.3f}  -> {[w.strip() for w in tok.batch_decode([[int(t)] for t in tks])]}')
feats = [c for c, _ in ranked]

In [ ]:
def collateral_line(rep, tag):
    m = rep.metrics().get(tag, {})
    n = int(m.get('n', 0)); p = m.get('moved', 0.0); dp = m.get('mean_dprob', 0.0)
    se = math.sqrt(p * (1 - p) / n) if n else float('nan')
    print(f'  {tag:<10} n={n:>3}  collateral rate {p:.3f} +/- {se:.3f} (1 s.e.)   mean dprob {dp:+.3f}')

for label, batt in [('NARROW', narrow_k), ('WIDE', wide_k)]:
    rep = marv.study_edit(model, tok, marv.suppress(model, feats[:5]), batt, device=device)
    print(f'--- {label} battery, suppress top 5 ---')
    for t in ('target', 'neighbour', 'control'):
        collateral_line(rep, t)
    print()

The `control` error bar collapses (~0.15 -> ~0.04) going narrow -> wide. With
the wide battery you can also read collateral **by sub-domain** below -- an edit
that quietly damages lexical knowledge looks very different from one that only
nudges geography.

In [ ]:
rep = marv.study_edit(model, tok, marv.suppress(model, feats[:5]), wide_k, device=device)
rep.show()
print()
for tag, m in sorted(rep.metrics().items()):
    if tag == '_all':
        continue
    print(f'  {tag:<14} n={int(m["n"]):>3}  moved={m["moved"]:.3f}  mean dprob={m["mean_dprob"]:+.3f}')

## Which layers carry the fact?

`suppression_by_layer` suppresses one layer's slice of the constellation at a
time. Layers with a `target` drop are load-bearing; layers with features in the
constellation but ~zero effect were geometric KNN hits.

In [ ]:
from marv.evaluate import suppression_by_layer
edit_feats = feats[:8]

print(f'{"L":>3} {"n":>2} {"features":>16}  {"target dp":>10} {"neigh dp":>10} {"ctrl dp":>10}')
iso = suppression_by_layer(model, tok, edit_feats, wide_k, device=device)
for L, fs, d in iso:
    m = d.metrics(); g = lambda t: m.get(t, {}).get('mean_dprob', 0.0)
    print(f'{L:>3} {len(fs):>2} {str(list(fs)):>16}  {g("target"):>+10.3f} {g("neighbour"):>+10.3f} {g("control"):>+10.3f}')

print('\ncumulative, shallow -> deep:')
for L, fs, d in suppression_by_layer(model, tok, edit_feats, wide_k, device=device, cumulative=True):
    m = d.metrics(); g = lambda t: m.get(t, {}).get('mean_dprob', 0.0)
    print(f'  <=L{L:<2}  target {g("target"):>+.3f}   neigh {g("neighbour"):>+.3f}   ctrl {g("control"):>+.3f}')

## The Pareto frontier

In [ ]:
import matplotlib.pyplot as plt
from marv.evaluate import frontier_table

sizes = [0, 1, 2, 3, 4, 6, 8, 10, 14]
sweep = marv.suppression_frontier(model, tok, feats, wide_k, sizes=sizes, device=device)
rows = list(frontier_table(sweep, target='target', collateral='neighbour'))
print(f'{"n":>3} {"target drop":>12} {"neigh drop":>12} {"neigh moved":>12}')
for n, td, cd, cm in rows:
    print(f'{n:>3} {td:>+12.3f} {cd:>+12.3f} {cm:>12.2f}')

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([r[2] for r in rows], [r[1] for r in rows], 'o-')
for n, td, cd, cm in rows:
    ax.annotate(str(n), (cd, td), fontsize=8, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('neighbour prob drop (collateral)'); ax.set_ylabel('target prob drop (efficacy)')
ax.set_title(f'AMD-135M: {COUNTRY}->{CAPITAL} suppression frontier'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Weight-space diff: base vs code finetune

`amd/AMD-Llama-135m-code` is the same 12-layer model finetuned on StarCoder
Python (20B tokens). `marv.diff` shows which FFN features that moved.

In [ ]:
del model; gc.collect()
if device == 'cuda':
    torch.cuda.empty_cache()

code_m = LlamaForCausalLM.from_pretrained(CODE, torch_dtype=torch.float32)
vindex_code = marv.extract(code_m, model_name=CODE)
marv.build_down_meta(vindex_code, device=device)
vindex_code.save('/content/amd-135m-code.vindex.npz')
del code_m; gc.collect()

deltas = marv.diff(vindex, vindex_code)
print('most-moved features, code finetuning:')
for d in marv.most_changed(deltas, k=12):
    b, _ = marv.describe_feature(vindex,      d.layer, d.feature_idx, k=3)
    a, _ = marv.describe_feature(vindex_code, d.layer, d.feature_idx, k=3)
    bw = [w.strip() for w in tok.batch_decode([[int(t)] for t in b])]
    aw = [w.strip() for w in tok.batch_decode([[int(t)] for t in a])]
    print(f'  L{d.layer:>2} f{d.feature_idx:<5} gate_cos={d.gate_cos_sim:+.3f} down_cos={d.down_cos_sim:+.3f}  {bw} -> {aw}')

scores = marv.per_layer_score(deltas, metric='mean_topk')
print('\nlayers that absorbed the most change:')
for L in sorted(scores, key=lambda l: -scores[l])[:6]:
    print(f'  L{L:>2}: {scores[L]:.4f}')

## What else you can do here

- **Swap the fact.** The candidate list at the top drives everything -- add
  countries, or hand-write target/neighbour probes for a non-capital fact.
- **Rare vs common.** Run the frontier for a common and a rarely-mentioned
  country; the rare one usually has a sharper knee (fewer shared features).
- **suppress vs ablate vs steer** on the same constellation (see the Qwen2.5 notebook).
- **`marv.diff(vindex, quantized_vindex)`** -- which features 4-bit breaks (needs a
  bitsandbytes NF4 load; AMD-135M is small enough to hold both copies anywhere).
- Reload any vindex with `marv.VindexLite.load('/content/amd-135m-*.vindex.npz')`.

## Refine the edit: suppress only the causally-cleanest layer

The top-5 edit knocked out **every** capital (`geo` moved ~54%) while leaving
science / commonsense / history untouched -- MARV located the model's generic
*"capital-of-a-country"* circuit, not a France-specific one. But
`suppression_by_layer` showed the constellation's **L9 slice** drops the target
while barely moving the neighbours, whereas the **L7 slice** is the blunt part
that hits every capital equally.

So: keep only the cleanest layer's features and re-measure. If the per-layer
read is right, `L9-only` should hold most of the France efficacy while cutting
the geography collateral roughly in half -- the per-layer analysis handing you a
sharper edit than the flat top-k ranking did.

In [ ]:
# cell above freed the base model -- reload it (vindex, tok, feats, wide_k survive)
from marv.evaluate import suppression_by_layer, frontier_table
model = LlamaForCausalLM.from_pretrained(BASE, torch_dtype=DTYPE).to(device).eval()

edit_feats = feats[:8]
iso = suppression_by_layer(model, tok, edit_feats, wide_k, device=device)

# pick the layer whose slice has the best target-drop : neighbour-drop ratio
def _clean_ratio(d):
    m = d.metrics()
    t  = -m.get('target',    {}).get('mean_dprob', 0.0)
    nb = -m.get('neighbour', {}).get('mean_dprob', 0.0)
    return t / nb if nb > 1e-6 else (float('inf') if t > 1e-6 else 0.0)

best_L, best_fs, _ = max(iso, key=lambda x: _clean_ratio(x[2]))
clean_feats = [(best_L, f) for f in best_fs]
print(f'cleanest layer: L{best_L}  features {list(best_fs)}\n')

print(f'{"edit":<15}{"target dp":>11}{"target mv":>11}{"neigh dp":>11}{"geo dp":>10}{"geo mv":>9}{"sci dp":>9}')
for label, ef in [('top-5 (mixed)', edit_feats[:5]), (f'L{best_L}-only', clean_feats)]:
    m  = marv.study_edit(model, tok, marv.suppress(model, ef), wide_k, device=device).metrics()
    g  = lambda t: m.get(t, {}).get('mean_dprob', 0.0)
    mv = lambda t: m.get(t, {}).get('moved', 0.0)
    print(f'{label:<15}{g("target"):>+11.3f}{mv("target"):>11.2f}{g("neighbour"):>+11.3f}'
          f'{g("geo"):>+10.3f}{mv("geo"):>9.2f}{g("science"):>+9.3f}')

# frontier for the clean subset -- efficacy vs geo collateral as n grows
print(f'\nfrontier, L{best_L}-only features:')
sweep = marv.suppression_frontier(model, tok, clean_feats, wide_k,
                                  sizes=list(range(len(clean_feats) + 1)), device=device)
for n, td, cd, cm in frontier_table(sweep, target='target', collateral='geo'):
    print(f'  n={n}  target drop {td:+.3f}   geo drop {cd:+.3f}   geo moved {cm:.2f}')